# League of Legends 👑
<p align="center">
    <img src="image/imrs.avif" width="900">
</p>

**Author**: Hsin Yu Ho

**Website Link**:  https://jasonyuki71.github.io/LoL-Turrets-and-Winrate/ 



In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

## Introduction
League of Legends (LoL) is a multiplayer online battle arena (MOBA) game developed by Riot Games. There are more than 150 million players playing this game, which is one of the world’s most popular esports.

The project uses 2022 League of Legends esports match dataset from Oracle’s Elixir. This dataset contains a series of key game statistics and results from League of Legends matches, providing insights into which factors, champion pairings, tactics, or other factors can lead to victory. It includes various features such as banned/ picked champions, in-game statistics, and so on. 

In League of Legends, teams win by destroying the enemy Nexus. To reach the Nexus, teams must first destroy defensive structures such as turrets and inhibitors across the map. Because turrets provide map control and protect important objectives, teams that destroy more turrets often gain significant strategic advantages. This project investigates how objective control, especially turret destruction, relates to winning professional League of Legends matches.


## Research Question

Do turret destructions have a stronger relationship with match outcomes than kills in professional League of Legends matches?

## Hypothesis
**Null Hypothesis**: Tower destructions and kills have the same relationship strength with match outcomes. Any observed difference between their relationships with winning is due to random chance.

**Alternative Hypothesis**: Tower destructions have a stronger relationship with match outcomes than kills.

**Test Statistic:** Difference in win rates between the tower-based comparison and the kill-based comparison.

## Data Cleaning and Exploratory Data Analysis
### Data Cleaning

In [21]:
lol = pd.read_csv('2022_LoL.csv')
lol.shape

/var/folders/kp/39p7dlxs58lgzznbm1_j69br0000gn/T/ipykernel_51724/677921804.py:1: DtypeWarning:

Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.



(150348, 165)

Keep only necessary columns:

In [22]:
lol_team = lol[lol['position'] == 'team'].copy()
lol_team = lol_team[['gameid', 'side', 'result', 'teamkills', 'teamdeaths', 'towers', 'opp_towers', 'dragons', 'barons', 'golddiffat25', 'xpdiffat25', 'csdiffat25']]
lol_team

,gameid,side,result,teamkills,teamdeaths,towers,opp_towers,dragons,barons,golddiffat25,xpdiffat25,csdiffat25
10,ESPORTSTMNT01_2690210,Blue,0,9,19,3.0,6.0,1.0,0.0,88.0,-3971.0,-97.0
11,ESPORTSTMNT01_2690210,Red,1,19,9,6.0,3.0,3.0,0.0,-88.0,3971.0,97.0
22,ESPORTSTMNT01_2690219,Blue,0,3,16,3.0,11.0,1.0,0.0,-7280.0,-7746.0,-33.0
23,ESPORTSTMNT01_2690219,Red,1,16,3,11.0,3.0,4.0,2.0,7280.0,7746.0,33.0
34,8401-8401_game_1,Blue,1,13,6,8.0,3.0,2.0,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
150323,9687-9687_game_3,Red,0,7,26,5.0,10.0,0.0,0.0,NaN,NaN,NaN
150334,9687-9687_game_4,Blue,1,19,7,11.0,2.0,4.0,2.0,NaN,NaN,NaN
150335,9687-9687_game_4,Red,0,7,19,2.0,11.0,1.0,0.0,NaN,NaN,NaN
150346,9687-9687_game_5,Blue,0,8,21,0.0,10.0,0.0,0.0,NaN,NaN,NaN


In [23]:
lol_team.head().to_html("assets/cleaned_df.html", index=False)

Check missing data:

In [24]:
lol_team.isna().sum()

gameid             0
side               0
result             0
teamkills          0
teamdeaths         0
towers             0
opp_towers         0
dragons            0
barons             0
golddiffat25    5110
xpdiffat25      5110
csdiffat25      5110
dtype: int64

In [25]:
lol_team.describe()

,result,teamkills,teamdeaths,towers,opp_towers,dragons,barons,golddiffat25,xpdiffat25,csdiffat25
count,25058.00000,25058.000000,25058.000000,25058.000000,25058.000000,25058.000000,25058.000000,19948.000000,19948.000000,19948.000000
mean,0.50000,14.490063,14.518956,6.041025,6.041025,2.231942,0.673278,0.000000,0.000000,0.000000
std,0.50001,7.531424,7.530013,3.616012,3.616012,1.374340,0.722837,6859.092178,6706.172939,70.032731
min,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-23344.000000,-20370.000000,-303.000000
25%,0.00000,8.000000,8.000000,3.000000,3.000000,1.000000,0.000000,-4783.750000,-4773.000000,-47.000000
50%,0.50000,14.000000,14.000000,7.000000,7.000000,2.000000,1.000000,0.000000,0.000000,0.000000
75%,1.00000,20.000000,20.000000,9.000000,9.000000,3.000000,1.000000,4783.750000,4773.000000,47.000000
max,1.00000,60.000000,60.000000,11.000000,11.000000,7.000000,4.000000,23344.000000,20370.000000,303.000000


In [26]:
lol_team[['towers', 'teamkills']].describe().to_html("assets/describe_towers_teamkills.html", index=True)

### Univariate Analysis

In [27]:
fig_towers = px.histogram(
    lol_team,
    x='towers',
    nbins=12,
    title='Distribution of Towers Destroyed'
)

fig_towers.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)

fig_towers.write_html(
    "assets/towers_histogram.html",
    include_plotlyjs="cdn"
)

fig_towers.show()

In [28]:
fig_kills = px.histogram(
    lol_team,
    x='teamkills',
    nbins=20,
    title='Distribution of Team Kills'
)
fig_kills.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)

fig_kills.write_html(
    "assets/kills_histogram.html",
    include_plotlyjs="cdn"
)

fig_kills.show()

### Bivariate Analysis

#### Towers vs Win

In [29]:
lol_team['match_result'] = lol_team['result'].map({0: 'Loss', 1: 'Win'})

fig_tower_win = px.box(
    lol_team,
    x='match_result',
    y='towers',
    title='Towers Destroyed vs Match Result',
    labels={
        'match_result': 'Match Result',
        'towers': 'Towers Destroyed'
    }
)
fig_tower_win.show()

fig_tower_win.write_html(
    "assets/towers_boxplot.html",
    include_plotlyjs="cdn"
)

#### Kills vs Win

In [30]:
fig_kills_win = px.box(
    lol_team,
    x='match_result',
    y='teamkills',
    title='Kills vs Match Result',
    labels={
        'match_result': 'Match Result',
        'teamkills': 'Team Kills'
    }
)
fig_kills_win.show()

fig_kills_win.write_html(
    "assets/kills_boxplot.html",
    include_plotlyjs="cdn"
)

### Interesting Aggregates
#### Pivot Table

In [31]:
pivot = pd.pivot_table(
    lol_team,
    values=['teamkills', 'towers'],
    index='match_result',
    aggfunc=np.mean
)
pivot

/var/folders/kp/39p7dlxs58lgzznbm1_j69br0000gn/T/ipykernel_51724/2853062489.py:1: FutureWarning:

The provided callable <function mean at 0x105922e60> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.



,teamkills,towers
match_result,,
Loss,9.370820,2.841887
Win,19.609306,9.240163


In [32]:
pivot.to_html("assets/pivot.html", index=True)

#### Correlation Table & Heatmap

In [33]:
corr = lol_team[['result', 'teamkills', 'towers']].corr()
corr

,result,teamkills,towers
result,1.000000,0.679731,0.884732
teamkills,0.679731,1.000000,0.711172
towers,0.884732,0.711172,1.000000


In [34]:
px.imshow(
    corr,
    text_auto=True,
    title='Correlation Heatmap'
)


## Assessment of Missingness

### NMAR Analysis
The missingness of the `url` column may depend on the missing values themselves or on information that is not observed in the dataset. Since this dependency cannot be verified directly using the available data, the missingness mechanism may be NMAR (Not Missing At Random).

### Missingness Dependency
The column `assistsat25` depends on `datacompleteness` because `datacompleteness` rows labeled as partial are much more likely to have missing values for `assistsat25`, while rows labeled as complete usually contain valid values. 

#### Permutation Test: Missingness of assistsat25 vs datacompleteness

- **Null Hypothesis**: The missingness of `assistsat25` does not depend on `datacompleteness`.

- **Alternative Hypothesis**: The missingness of `assistsat25` does depend on `datacompleteness`.

In [35]:
lol['assists_nan'] = lol['assistsat25'].isna()

In [36]:
obs = (
    lol.pivot_table(
        index='assists_nan',
        columns='datacompleteness',
        aggfunc='size',
        observed=False
    )
    .apply(lambda x: x / x.sum(), axis=1)
)

obs_tvd = (obs.iloc[0] - obs.iloc[1]).abs().sum() / 2
print('Observed TVD:', obs_tvd)

Observed TVD: 0.3694716242661448


In [37]:
tvds = []

for _ in range(500):
    shuffled = np.random.permutation(lol['assists_nan'])
    shuffled_df = (
        lol.assign(shuffled=shuffled)
        .pivot_table(
            index='shuffled',
            columns='datacompleteness',
            aggfunc='size',
            observed=False
        )
        .apply(lambda x: x / x.sum(), axis=1)
    )

    tvd = (shuffled_df.iloc[0] - shuffled_df.iloc[1]).abs().sum() / 2
    tvds.append(tvd)

p_value = np.mean(np.array(tvds) >= obs_tvd)
print('p-value:', p_value)

p-value: 0.0


In [38]:
fig = px.histogram(
    x=tvds,
    nbins=30,
    title='Permutation Distribution of TVD (Missingness assistsat25 vs DataComplete)',
    labels={'x': 'TVD'}
)
fig.add_vline(
    x=obs_tvd,
    line_color='red',
    annotation_text='Observed TVD'
)
fig.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)
fig.show()

fig.write_html(
    'assets/missingness_assistsat25_DataComplete.html',
    include_plotlyjs='cdn'
)

The p-value from the permutation test is extremely small and close to 0. This indicates that the observed TVD is very unlikely to occur under the null hypothesis. Therefore, there is strong evidence that the missingness of assistsat25 depends on datacompleteness, we reject null hypothesis.

#### Permutation Test: Missingness of assistsat25 vs datacompleteness
The variable `assistsat25` does not depend on `side` because whether a team is blue side or red side should NOT affect whether `assistsat25` is missing.

- **Null Hypothesis**: The missingness of `assistsat25` does not depend on `side`.

- **Alternative Hypothesis**: The missingness of `assistsat25` does depend on `side`.

In [39]:
obs = (
    lol.pivot_table(
        index='assists_nan',
        columns='side',
        aggfunc='size',
        observed=False
    )
    .apply(lambda x: x / x.sum(), axis=1)
)

obs_tvd = (obs.iloc[0] - obs.iloc[1]).abs().sum() / 2
print('Observed TVD:', obs_tvd)

Observed TVD: 0.0


In [40]:
tvds = []

for _ in range(500):
    shuffled = np.random.permutation(lol['assists_nan'])
    shuffled_df = (
        lol.assign(shuffled=shuffled)
        .pivot_table(
            index='shuffled',
            columns='side',
            aggfunc='size',
            observed=False
        )
        .apply(lambda x: x / x.sum(), axis=1)
    )

    tvd = (shuffled_df.iloc[0] - shuffled_df.iloc[1]).abs().sum() / 2
    tvds.append(tvd)

p_value = np.mean(np.array(tvds) >= obs_tvd)
print('p-value:', p_value)

p-value: 1.0


In [41]:
fig = px.histogram(
    x=tvds,
    nbins=30,
    title='Permutation Distribution of TVD (Missingness assistsat25 vs Side)',
    labels={'x': 'TVD'},
    
)

fig.add_vline(
    x=obs_tvd,
    line_color='red',
    annotation_text='Observed TVD'
)

fig.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)
fig.show()

fig.write_html(
    'assets/missingness_assistsat25_side.html',
    include_plotlyjs='cdn'
)

The observed TVD between the distributions of `side` for rows where `assistsat25` is missing and rows where it is not missing was 0. The permutation test produced a p-value of 1.0.

p-value is much greater than 0.05, so fail to reject the null hypothesis. There is no evidence that the missingness of `assistsat25` depends on a team's side (Blue/Red).

In fact, the observed TVD of 0 indicates that the distribution of `side` is identical for missing and non-missing values of `assistsat25` in our dataset. Therefore, the missingness of `assistsat25` appears to be independent of `side`.


## Hypothesis Testing
- **Null Hypothesis**: Tower destructions and kills have the same relationship strength with match outcomes. Any observed difference between their relationships with winning is due to random chance.

- **Alternative Hypothesis**: Tower destructions have a stronger relationship with match outcomes than kills.

- **Test Statistic:** Difference in win rates between the tower-based comparison and the kill-based comparison.

I will use **permutation test** with **p-value** to see if turret destructions are more likely to win than teams with more kills or not.

In [42]:
lol_team['more_towers'] = lol_team['towers'] > lol_team['opp_towers']
lol_team['more_kills'] = lol_team['teamkills'] > lol_team['teamdeaths']

# Win rate with more towers
tower_win = lol_team.loc[lol_team['more_towers'], 'result'].mean()
# Win rate with more teamkills
kill_win = lol_team.loc[lol_team['more_kills'], 'result'].mean()
observed = tower_win - kill_win

print('Observed difference:', observed)

Observed difference: 0.020334582025422487


In [43]:
diffs = []

for _ in range(1000):
    shuffle = np.random.permutation(lol_team['result'])
    shuffle_df = lol_team.assign(shuffled_result=shuffle)
    tower_rate = shuffle_df.loc[shuffle_df['more_towers'], 'shuffled_result'].mean()
    kill_rate = shuffle_df.loc[shuffle_df['more_kills'], 'shuffled_result'].mean()
    diffs.append(tower_rate - kill_rate)

p_value = np.mean(np.array(diffs) >= observed)
print('p-value:', p_value)

p-value: 0.0


In [44]:
fig = px.histogram(
    x=diffs,
    nbins=30,
    title='Permutation Distribution of Difference in Win Rates',
    labels={'x': 'Difference in Win Rate'}
)

fig.add_vline(
    x=observed,
    line_color='red',
    annotation_text='Observed Statistic'
)

fig.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)

fig.show()

fig.write_html(
    'assets/hypothesis_test.html',
    include_plotlyjs='cdn'
)

This provides evidence that teams with more destroyed towers have a higher win rate than teams with more kills in this dataset.

## Problem Identification

The prediction problem in this project is to determine whether a team will win a professional League of Legends match using in-game statistics such as turret destructions, kills, objectives, and side selection.

The response variable is `result`, which has two possible outcomes:

* `1` = win
* `0` = loss

Because the response variable has two categories, this is a binary classification problem.

I use accuracy as the evaluation metric because the dataset is relatively balanced between wins and losses at the team level, and the goal is to measure how often the model correctly predicts the match result overall.

At the time of prediction, I assume that the model is being used after or near the end of a match, when team-level statistics such as towers destroyed, kills, dragons, barons, gold difference, XP difference, CS difference, and side selection are known. I will not use features that directly reveal the result itself, such as the `result` column, as predictors.

This prediction problem aims to identify which in-game statistics are most predictive of winning and to better understand the factors associated with success in professional League of Legends matches. It also aligns with the overall theme of this project, which investigates whether turret destruction is more strongly associated with winning than kills.

## Baseline Model
For my baseline model, I used **Logistic Regression** with three features: `towers`, `teamkills`, and `side`. The variables `towers` and `teamkills` are numerical and were left unchanged, while the categorical variable `side` was transformed using **One-Hot Encoding**. All preprocessing and model training steps were implemented within a single sklearn Pipeline.

I evaluated the model using a train-test split to assess its ability to generalize to unseen matches. Logistic Regression was chosen because it is a simple and interpretable classification model commonly used as a baseline for binary prediction tasks.

In [45]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


X = lol_team[['towers', 'teamkills', 'side']]
y = lol_team['result']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

preproc = ColumnTransformer(
    [
        ('onehot', OneHotEncoder(drop='if_binary'), ['side'])
    ],
    remainder='passthrough'
)

baseline_model = Pipeline([
    ('preproc', preproc),
    ('clf', LogisticRegression(max_iter=1000))
])

baseline_model.fit(X_train, y_train)

train_acc = baseline_model.score(X_train, y_train)
test_acc = baseline_model.score(X_test, y_test)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 0.9655190762517959
Test Accuracy: 0.965682362330407


The baseline Logistic Regression model achieved a training accuracy of 96.55% and a test accuracy of 96.57%. The difference between train and test accuracy is very small, suggesting that the model generalizes well and is not overfitting the training data. However, the model uses only a small number of features and may not fully capture other important aspects of team performance, such as objective control and relative advantages over the opponent.

## Final Model
For final model, I will improve on the baseline model by adding features that better represent a team's advantage over the opponent, not just the team's raw statistics.

The baseline model only used raw `towers`, raw `teamkills`, and `side`. For the final model, I will add these new engineered features:

* `tower_advantage = towers - opp_towers`: This measures how many more towers a team destroyed compared to its opponent. Since League of Legends is a two-team game, the difference between a team and its opponent should be more meaningful than the raw number of towers alone.
* `kill_death_ratio = (teamkills + 1) / (teamdeaths + 1)`: This measures combat efficiency. I add 1 to avoid division by zero.
* `objective_score = dragons + 2 * barons`: Dragons and barons are important neutral objectives. I weight barons more heavily because baron is usually a stronger late-game objective.

I use a **Random Forest Classifier** as the final model because it captures nonlinear relationships and interactions between features. This is useful because the effect of towers, kills, and objectives on winning may not be perfectly linear.

### Hyperparameter Tuning
I used GridSearchCV with 5-fold cross-validation to tune the following hyperparameters:
* `n_estimators`: the number of decision trees in the forest. More trees can make predictions more stable. [100, 200]
* `max_depth`: controls how deep each tree can grow. This helps control overfitting. [3, 5, 8, None]
* `min_samples_leaf`: controls the minimum number of samples required at a leaf node. Larger values can make the model less overfit. [1, 5, 10]

In [46]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Use the same rows and same random_state as the baseline model.
# This keeps the final model's test set directly comparable to the baseline model.
final_cols = [
    'towers', 'opp_towers',
    'teamkills', 'teamdeaths',
    'dragons', 'barons',
    'side'
]

X_final = lol_team[final_cols]
y_final = lol_team['result']

X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final,
    y_final,
    test_size=0.25,
    random_state=42
)

def add_features(data):
    data = data.copy()
    data['tower_advantage'] = data['towers'] - data['opp_towers']
    data['kill_death_ratio'] = (data['teamkills'] + 1) / (data['teamdeaths'] + 1)
    data['objective_score'] = data['dragons'] + 2 * data['barons']

    return data

In [47]:
numeric_features = [
    'towers', 'opp_towers',
    'teamkills', 'teamdeaths',
    'dragons', 'barons',
    'tower_advantage', 'kill_death_ratio', 'objective_score'
]
categorical_features = ['side']

final_preproc = ColumnTransformer(
    [('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), categorical_features),
     ('num', SimpleImputer(strategy='median'), numeric_features)],
    remainder='drop'
)

final_pipeline = Pipeline([
    ('engineer_features', FunctionTransformer(add_features, validate=False)),
    ('preproc', final_preproc),
    ('clf', RandomForestClassifier(random_state=42))
])

param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [3, 5, 8, None],
    'clf__min_samples_leaf': [1, 5, 10]
}

grid_search = GridSearchCV(
    estimator=final_pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train_final, y_train_final)

print("Best Hyperparameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

Best Hyperparameters: {'clf__max_depth': None, 'clf__min_samples_leaf': 5, 'clf__n_estimators': 100}
Best CV Accuracy: 0.9852070482323707


In [48]:
final_model = grid_search.best_estimator_
final_train_acc = final_model.score(X_train_final, y_train_final)
final_test_acc = final_model.score(X_test_final, y_test_final)

print("Baseline Train Accuracy:", train_acc)
print("Baseline Test Accuracy:", test_acc)
print("Final Train Accuracy:", final_train_acc)
print("Final Test Accuracy:", final_test_acc)
print("Improvement in Test Accuracy:", final_test_acc - test_acc)

Baseline Train Accuracy: 0.9655190762517959
Baseline Test Accuracy: 0.965682362330407
Final Train Accuracy: 0.9898366413026127
Final Test Accuracy: 0.9854748603351955
Improvement in Test Accuracy: 0.01979249800478844


The final model achieved a training accuracy of 98.98% and a testing accuracy of 98.55%, improving upon the baseline model's testing accuracy of 96.57%. This improvement suggests that the engineered features and Random Forest classifier were able to capture additional information about match outcomes that was not utilized by the baseline Logistic Regression model.

Although the final model achieved a higher training accuracy than the baseline model, the testing accuracy also increased substantially. The difference between training and testing accuracy is only about 0.43 percentage points, indicating little evidence of severe overfitting and suggesting that the model generalizes well to unseen matches.

In [49]:
from sklearn.metrics import confusion_matrix


y_pred_final = final_model.predict(X_test_final)

cm = confusion_matrix(y_test_final, y_pred_final)

cm_df = pd.DataFrame(
    cm,
    index=['Actual Loss (0)', 'Actual Win (1)'],
    columns=['Predicted Loss (0)', 'Predicted Win (1)']
)

fig_cm = px.imshow(
    cm_df,
    text_auto=True,
    title='Confusion Matrix of Final Random Forest Model',
    labels=dict(x='Predicted Result', y='Actual Result', color='Count')
)

fig_cm.show()

fig_cm.write_html(
    'assets/confusion_matrix.html',
    include_plotlyjs='cdn'
)

## Fairness Analysis
To evaluate the fairness of the final model, I will compare its predictive performance on teams playing on the Blue Side versus teams playing on the Red Side.

### Hypotheses
- **Null Hypothesis:** The model is fair. Any difference in accuracy between Blue Side and Red Side teams is due to random chance.

- **Alternative Hypothesis:** The model is unfair. The model's accuracy differs between Blue Side and Red Side teams.

In [51]:
y_pred = final_model.predict(X_test_final)

fairness_df = X_test_final.copy()
fairness_df['actual'] = y_test_final.values
fairness_df['predicted'] = y_pred
fairness_df['correct'] = fairness_df['actual'] == fairness_df['predicted']

In [52]:
blue_acc = fairness_df.loc[fairness_df['side']=='Blue', 'correct'].mean()
red_acc = fairness_df.loc[fairness_df['side']=='Red', 'correct'].mean()

observed_diff = abs(blue_acc - red_acc)

print('Blue Accuracy:', blue_acc)
print('Red Accuracy:', red_acc)
print('Observed Difference:', observed_diff)

Blue Accuracy: 0.9850318471337579
Red Accuracy: 0.98592
Observed Difference: 0.000888152866242109


In [53]:
n_reps = 1000
diffs = []

for _ in range(n_reps):
    shuffled = fairness_df['side'].sample(frac=1, replace=False).values

    temp = fairness_df.copy()
    temp['shuffled_side'] = shuffled

    blue = temp.loc[temp['shuffled_side']=='Blue', 'correct'].mean()
    red = temp.loc[temp['shuffled_side']=='Red', 'correct'].mean()

    diffs.append(abs(blue - red))

p_value = np.mean(np.array(diffs) >= observed_diff)
print('p-value:', p_value)

p-value: 0.818


In [54]:
perm_df = pd.DataFrame({'Difference in Accuracy': diffs})
fig = px.histogram(
    perm_df,
    x='Difference in Accuracy',
    nbins=30,
    title='Permutation Test for Fairness'
)
fig.add_vline(
    x=observed_diff,
    line_color='red',
    line_width=3
)
fig.update_traces(
    marker_line_color='white',
    marker_line_width=0.1
)
fig.show()

fig.write_html(
    'assets/fairness_test.html',
    include_plotlyjs='cdn'
)

The observed difference (red line) lies near the center of the permutation distribution rather than in the extreme tail, providing visual evidence that the observed difference is consistent with random chance.

The permutation test produced a p-value of 0.804. Since this p-value is substantially greater than the significance level of 0.05, we fail to reject the null hypothesis.

Therefore, there is insufficient evidence that the model performs differently for Blue Side and Red Side teams. The observed difference(0.00089) in accuracy is very small and is likely attributable to random chance. Based on this analysis, the final model appears to be fair with respect to side selection.